In [ ]:
# ============================================================
# STEP 14.1 — RESTORE PHASE 14
# ============================================================

from google.colab import drive

import os
import json
import joblib
import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT_PATH = (
    "/content/drive/MyDrive/Delhi-AQI-Prediction"
)

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

model_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_10_station_feature_engineered.parquet"
    )
)

model_df["Datetime"] = pd.to_datetime(
    model_df["Datetime"]
)

model_df["StationId"] = (
    model_df["StationId"].astype(str)
)

model_df = (
    model_df
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

tuned_rf = joblib.load(
    os.path.join(
        MODEL_PATH,
        "phase13B_tuned_random_forest.joblib"
    )
)

print("=" * 60)
print("PHASE 14 RESTORED")
print("=" * 60)

print(
    "Rows:",
    len(model_df)
)

print(
    "Stations:",
    model_df["StationId"].nunique()
)

print(
    "Current champion:",
    "Tuned Random Forest"
)

In [ ]:
# ============================================================
# STEP 14.2 — FEATURE GROUP DEFINITIONS
# ============================================================

RAW_FEATURES = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

TARGET = "AQI_t_plus_1"

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    f"{column}_lag_{lag}h"
    for lag in [1, 2, 12]
    for column in RAW_FEATURES
]

rolling_features = [
    "AQI_rolling_mean_3h",
    "AQI_rolling_std_3h",
    "AQI_rolling_mean_12h",
    "AQI_rolling_std_12h",

    "PM2.5_rolling_mean_3h",
    "PM2.5_rolling_std_3h",
    "PM2.5_rolling_mean_12h",
    "PM2.5_rolling_std_12h",

    "PM10_rolling_mean_3h",
    "PM10_rolling_std_3h",
    "PM10_rolling_mean_12h",
    "PM10_rolling_std_12h",

    "NO2_rolling_mean_3h",
    "NO2_rolling_std_3h",
    "NO2_rolling_mean_12h",
    "NO2_rolling_std_12h"
]

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

station_features = [
    "StationId"
]

feature_groups = {
    "Current": current_features,
    "Lag": lag_features,
    "Rolling": rolling_features,
    "Calendar": calendar_features,
    "Events": event_features,
    "Station": station_features
}

full_features = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
    + station_features
)

missing = [
    feature
    for feature in full_features
    if feature not in model_df.columns
]

assert not missing, (
    f"Missing features: {missing}"
)

print(
    "Full feature count:",
    len(full_features)
)

for group_name, features in feature_groups.items():
    print(
        f"{group_name}: {len(features)}"
    )

In [ ]:
# ============================================================
# STEP 14.3 — FIXED FINAL TEST PERIOD
# ============================================================

unique_times = np.sort(
    model_df["Datetime"].unique()
)

split_position = int(
    len(unique_times) * 0.80
)

final_test_start = unique_times[
    split_position
]

optimization_df = (
    model_df[
        model_df["Datetime"] < final_test_start
    ]
    .copy()
    .reset_index(drop=True)
)

final_test_df = (
    model_df[
        model_df["Datetime"] >= final_test_start
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Optimization rows:",
    len(optimization_df)
)

print(
    "Final test rows:",
    len(final_test_df)
)

print(
    "Optimization end:",
    optimization_df["Datetime"].max()
)

print(
    "Final test start:",
    final_test_df["Datetime"].min()
)

assert (
    optimization_df["Datetime"].max()
    <
    final_test_df["Datetime"].min()
)

In [ ]:
# ============================================================
# STEP 14.4 — PROMOTED RF CONFIGURATION
# ============================================================

with open(
    os.path.join(
        OUTPUT_PATH,
        "phase13B_summary.json"
    ),
    "r"
) as f:
    phase13B_summary = json.load(f)

print(
    "Promoted model:",
    phase13B_summary["selected_model"]
)

# The actual promoted model was saved already.
# Extract its fitted estimator settings directly.
rf_estimator = (
    tuned_rf.named_steps["model"]
)

rf_params = rf_estimator.get_params()

selected_rf_params = {
    "n_estimators": rf_params["n_estimators"],
    "max_depth": rf_params["max_depth"],
    "min_samples_leaf": rf_params["min_samples_leaf"],
    "max_features": rf_params["max_features"],
    "random_state": rf_params["random_state"],
    "n_jobs": rf_params["n_jobs"]
}

print(
    selected_rf_params
)

In [ ]:
# ============================================================
# STEP 14.5 — PREPROCESSOR FACTORY
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

def make_preprocessor(
    numeric_columns,
    categorical_columns
):
    transformers = []

    if len(numeric_columns) > 0:

        numeric_transformer = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    )
                )
            ]
        )

        transformers.append(
            (
                "numeric",
                numeric_transformer,
                numeric_columns
            )
        )

    if len(categorical_columns) > 0:

        categorical_transformer = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False
                    )
                )
            ]
        )

        transformers.append(
            (
                "categorical",
                categorical_transformer,
                categorical_columns
            )
        )

    return ColumnTransformer(
        transformers=transformers
    )

In [ ]:
# ============================================================
# STEP 14.6 — TIME-AWARE ABLATION EVALUATOR
# ============================================================

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

optimization_times = np.sort(
    optimization_df["Datetime"].unique()
)

time_splitter = TimeSeriesSplit(
    n_splits=4
)


def evaluate_feature_set(
    feature_list,
    feature_set_name
):

    numeric_columns = [
        column
        for column in feature_list
        if column != "StationId"
    ]

    categorical_columns = [
        column
        for column in feature_list
        if column == "StationId"
    ]

    fold_results = []

    for fold, (
        train_idx,
        valid_idx
    ) in enumerate(
        time_splitter.split(
            optimization_times
        ),
        start=1
    ):

        train_times = optimization_times[
            train_idx
        ]

        valid_times = optimization_times[
            valid_idx
        ]

        fold_train = optimization_df[
            optimization_df["Datetime"].isin(
                train_times
            )
        ]

        fold_valid = optimization_df[
            optimization_df["Datetime"].isin(
                valid_times
            )
        ]

        X_train = fold_train[
            feature_list
        ]

        y_train = fold_train[
            TARGET
        ]

        X_valid = fold_valid[
            feature_list
        ]

        y_valid = fold_valid[
            TARGET
        ]

        preprocessor = make_preprocessor(
            numeric_columns,
            categorical_columns
        )

        model = Pipeline(
            steps=[
                (
                    "preprocessor",
                    preprocessor
                ),
                (
                    "model",
                    RandomForestRegressor(
                        **selected_rf_params
                    )
                )
            ]
        )

        model.fit(
            X_train,
            y_train
        )

        prediction = model.predict(
            X_valid
        )

        y_true = y_valid.to_numpy(
            dtype=float
        )

        y_pred = np.asarray(
            prediction,
            dtype=float
        )

        valid_mask = (
            np.isfinite(y_true)
            &
            np.isfinite(y_pred)
        )

        y_true = y_true[
            valid_mask
        ]

        y_pred = y_pred[
            valid_mask
        ]

        fold_results.append({
            "Fold": fold,

            "MAE": mean_absolute_error(
                y_true,
                y_pred
            ),

            "RMSE": np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred
                )
            ),

            "R2": r2_score(
                y_true,
                y_pred
            ),

            "Accuracy_±10_AQI_%": (
                np.mean(
                    np.abs(
                        y_true - y_pred
                    ) <= 10
                )
                * 100
            )
        })

    fold_df = pd.DataFrame(
        fold_results
    )

    return {
        "FeatureSet": feature_set_name,
        "FeatureCount": len(feature_list),
        "MAE": fold_df["MAE"].mean(),
        "RMSE": fold_df["RMSE"].mean(),
        "R2": fold_df["R2"].mean(),
        "Accuracy_±10_AQI_%":
            fold_df[
                "Accuracy_±10_AQI_%"
            ].mean()
    }

In [ ]:
# ============================================================
# STEP 14.7 — FULL FEATURE SET
# ============================================================

ablation_candidates = []

ablation_candidates.append(
    evaluate_feature_set(
        full_features,
        "Full Feature Set"
    )
)

print(
    "Full feature set evaluated."
)

In [ ]:
# ============================================================
# STEP 14.8 — ONE-GROUP-REMOVED ABLATIONS
# ============================================================

for group_name in feature_groups.keys():

    if group_name == "Station":
        # Keep station identity for all primary experiments.
        continue

    reduced_features = [
        feature
        for feature in full_features
        if feature not in feature_groups[
            group_name
        ]
    ]

    result = evaluate_feature_set(
        reduced_features,
        f"Without {group_name}"
    )

    ablation_candidates.append(
        result
    )

    print(
        f"Completed: Without {group_name}"
    )

In [ ]:
# ============================================================
# STEP 14.9 — TARGETED FEATURE COMBINATIONS
# ============================================================

targeted_sets = {

    "Current + Lag":
        current_features
        + lag_features
        + station_features,

    "Current + Rolling":
        current_features
        + rolling_features
        + station_features,

    "Current + Lag + Rolling":
        current_features
        + lag_features
        + rolling_features
        + station_features,

    "Current + Lag + Rolling + Calendar":
        current_features
        + lag_features
        + rolling_features
        + calendar_features
        + station_features,

    "AQI + Lag + Rolling":
        [
            "AQI"
        ]
        + lag_features
        + rolling_features
        + station_features
}

for name, feature_list in targeted_sets.items():

    result = evaluate_feature_set(
        list(dict.fromkeys(feature_list)),
        name
    )

    ablation_candidates.append(
        result
    )

    print(
        f"Completed: {name}"
    )

In [ ]:
# ============================================================
# PHASE 14 CHECKPOINT — STOPPED AFTER STEP 14.8
# ============================================================

import os
import json
from datetime import datetime

# Project paths
PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "checkpoints")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Checkpoint information
checkpoint = {
    "project": "Delhi AQI t+1 Hour Forecasting",
    "phase": "Phase 14 — Feature Optimization & Ablation",
    "completed_until": "14.8",
    "resume_from": "14.9",

    "forecast_target": "AQI_t_plus_1",
    "stations": 10,

    "current_champion": "Tuned Random Forest",
    "champion_rmse": 8.386059444844367,
    "champion_accuracy_plus_minus_10": 93.77083219496203,

    "completed_steps": [
        "14.1",
        "14.2",
        "14.3",
        "14.4",
        "14.5",
        "14.6",
        "14.7",
        "14.8"
    ],

    "next_step": "14.9",
    "status": "STOPPED — SAFE TO DISCONNECT RUNTIME",
    "timestamp": datetime.now().isoformat()
}

# Save checkpoint
checkpoint_path = os.path.join(
    OUTPUT_DIR,
    "phase14_checkpoint_step14_8.json"
)

with open(checkpoint_path, "w") as f:
    json.dump(checkpoint, f, indent=4)

# Also save a general session status file
session_status_path = os.path.join(
    OUTPUT_DIR,
    "session_status.json"
)

with open(session_status_path, "w") as f:
    json.dump(checkpoint, f, indent=4)

print("=" * 60)
print("CHECKPOINT SAVED SUCCESSFULLY")
print("=" * 60)
print(f"Completed until : Phase 14.8")
print(f"Resume from     : Phase 14.9")
print(f"Champion        : Tuned Random Forest")
print(f"Champion RMSE   : {checkpoint['champion_rmse']:.6f}")
print(f"Accuracy ±10    : {checkpoint['champion_accuracy_plus_minus_10']:.2f}%")
print(f"Checkpoint file : {checkpoint_path}")
print("=" * 60)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

print("Drive mounted.")
print("Project root:", PROJECT_ROOT)

In [ ]:
import os
import json

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/Delhi-AQI-Prediction/"
    "outputs/checkpoints/phase14_checkpoint_step14_8.json"
)

with open(CHECKPOINT_PATH, "r") as f:
    checkpoint = json.load(f)

print("=" * 60)
print("PHASE 14 CHECKPOINT RESTORED")
print("=" * 60)
print("Completed until :", checkpoint["completed_until"])
print("Resume from     :", checkpoint["resume_from"])
print("Champion        :", checkpoint["current_champion"])
print("Champion RMSE   :", checkpoint["champion_rmse"])
print("Accuracy ±10    :", checkpoint["champion_accuracy_plus_minus_10"])
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 14.9 — RESTORE FEATURE GROUP DEFINITIONS
# Run this after reconnecting Colab, before Step 14.9
# ============================================================

# Final raw features used for AQI(t) -> AQI(t+1)
RAW_FEATURES = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

TARGET = "AQI_t_plus_1"

# ------------------------------------------------------------
# Feature groups from Phase 14
# ------------------------------------------------------------

current_features = RAW_FEATURES.copy()

lag_features = [
    f"{feature}_lag{lag}"
    for feature in RAW_FEATURES
    for lag in [1, 2, 12]
]

rolling_features = [
    f"{feature}_roll{window}_{stat}"
    for feature in ["AQI", "PM2.5", "PM10", "NO2"]
    for window in [3, 12]
    for stat in ["mean", "std"]
]

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

station_features = [
    "StationId"
]

# ------------------------------------------------------------
# Combined feature groups
# ------------------------------------------------------------

feature_groups = {
    "Current": current_features,
    "Lag": lag_features,
    "Rolling": rolling_features,
    "Calendar": calendar_features,
    "Events": event_features,
    "Station": station_features
}

# Complete feature list
model_features = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
    + station_features
)

# Remove accidental duplicates while preserving order
model_features = list(dict.fromkeys(model_features))

print("=" * 60)
print("PHASE 14 FEATURE CONTRACT RESTORED")
print("=" * 60)

for group, features in feature_groups.items():
    print(f"{group:10s}: {len(features)} features")

print("-" * 60)
print("Total model features:", len(model_features))
print("Target:", TARGET)
print("=" * 60)

In [ ]:
# ============================================================
# DIAGNOSTIC — CHECK ACTUAL PHASE 14 FEATURE COLUMN NAMES
# ============================================================

import pandas as pd
import os

DATA_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction/data/processed/delhi_10_station_feature_engineered.parquet"

df_check = pd.read_parquet(DATA_PATH)

print("=" * 70)
print("DATASET CHECK")
print("=" * 70)
print("Shape:", df_check.shape)
print("Total columns:", len(df_check.columns))

print("\n--- Lag-related columns ---")
lag_cols = [
    c for c in df_check.columns
    if "lag" in c.lower()
]

print("Count:", len(lag_cols))
print(lag_cols)

print("\n--- Rolling-related columns ---")
roll_cols = [
    c for c in df_check.columns
    if "roll" in c.lower()
    or "rolling" in c.lower()
]

print("Count:", len(roll_cols))
print(roll_cols)

print("\n--- Calendar/Event columns ---")
special_cols = [
    c for c in df_check.columns
    if c in [
        "Hour", "Day", "Month", "Year",
        "DayOfWeek", "IsWeekend",
        "IsStubbleSeason", "IsDiwaliWindow",
        "StationId", "AQI_t_plus_1"
    ]
]

print(special_cols)

print("\n--- First 100 columns ---")
print(df_check.columns.tolist()[:100])

print("=" * 70)

In [ ]:
# ============================================================
# PHASE 14.9 — CORRECTED RESTORE + FEATURE ABLATION EVALUATOR
# Uses the ACTUAL saved Phase 14 column names
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "delhi_10_station_feature_engineered.parquet"
)

TUNED_RF_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "phase13B_tuned_random_forest.joblib"
)


# ------------------------------------------------------------
# 2. LOAD FEATURE-ENGINEERED DATA
# ------------------------------------------------------------

df14 = pd.read_parquet(DATA_PATH)

df14["Datetime"] = pd.to_datetime(
    df14["Datetime"],
    errors="coerce"
)

df14 = df14.dropna(
    subset=["Datetime", "AQI_t_plus_1"]
).copy()

df14 = df14.sort_values(
    ["Datetime", "StationId"]
).reset_index(drop=True)


# ------------------------------------------------------------
# 3. ACTUAL FEATURE GROUPS IN SAVED DATASET
# ------------------------------------------------------------

RAW_FEATURES = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

TARGET = "AQI_t_plus_1"


# Current features = t
current_features = RAW_FEATURES.copy()


# Lag features = t-1, t-2, t-12
lag_features = [
    f"{feature}_lag_{lag}h"
    for lag in [1, 2, 12]
    for feature in RAW_FEATURES
]


# Rolling features
rolling_features = [
    f"{feature}_rolling_{stat}_{window}h"
    for feature in [
        "AQI",
        "PM2.5",
        "PM10",
        "NO2"
    ]
    for window in [3, 12]
    for stat in ["mean", "std"]
]


# Calendar features
calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]


# Event features
event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]


# Station feature
station_features = [
    "StationId"
]


# ------------------------------------------------------------
# 4. FEATURE GROUP DICTIONARY
# ------------------------------------------------------------

feature_groups = {
    "Current": current_features,
    "Lag": lag_features,
    "Rolling": rolling_features,
    "Calendar": calendar_features,
    "Events": event_features,
    "Station": station_features
}


# Complete feature set
model_features = list(
    dict.fromkeys(
        current_features
        + lag_features
        + rolling_features
        + calendar_features
        + event_features
        + station_features
    )
)


# ------------------------------------------------------------
# 5. VERIFY EVERY FEATURE EXISTS
# ------------------------------------------------------------

missing_features = [
    f for f in model_features
    if f not in df14.columns
]

if missing_features:
    raise ValueError(
        "Unexpected missing features:\n"
        + "\n".join(missing_features)
    )


# ------------------------------------------------------------
# 6. LOAD TUNED RF PARAMETERS
# ------------------------------------------------------------

tuned_object = joblib.load(TUNED_RF_PATH)

if hasattr(tuned_object, "get_params"):
    saved_params = tuned_object.get_params()
else:
    saved_params = {}


rf_defaults = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
).get_params()

rf_params = {
    key: value
    for key, value in saved_params.items()
    if key in rf_defaults
}

# Ensure required settings
rf_params["random_state"] = 42
rf_params["n_jobs"] = -1


# Fallback if the saved artifact is not a standard RF object
if not rf_params:
    rf_params = {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 1.0,
        "random_state": 42,
        "n_jobs": -1
    }


# ------------------------------------------------------------
# 7. BUILD LEAKAGE-SAFE RF PIPELINE
# ------------------------------------------------------------

def build_rf_pipeline(feature_list):

    X = df14[feature_list]

    categorical_features = [
        col for col in feature_list
        if (
            X[col].dtype == "object"
            or str(X[col].dtype).startswith("category")
        )
    ]

    numeric_features = [
        col for col in feature_list
        if col not in categorical_features
    ]

    transformers = []

    if numeric_features:

        numeric_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            )
        ])

        transformers.append(
            (
                "numeric",
                numeric_pipeline,
                numeric_features
            )
        )

    if categorical_features:

        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ])

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    model = RandomForestRegressor(
        **rf_params
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])


# ------------------------------------------------------------
# 8. FEATURE-SET EVALUATOR
# ------------------------------------------------------------

def evaluate_feature_set(feature_list, feature_set_name):

    feature_list = list(
        dict.fromkeys(feature_list)
    )

    work = df14[
        ["Datetime", TARGET] + feature_list
    ].copy()

    work = work.replace(
        [np.inf, -np.inf],
        np.nan
    )

    work[TARGET] = pd.to_numeric(
        work[TARGET],
        errors="coerce"
    )

    work = work.dropna(
        subset=[TARGET]
    )

    work = work.sort_values(
        "Datetime"
    ).reset_index(drop=True)

    X = work[feature_list]
    y = work[TARGET]

    tscv = TimeSeriesSplit(
        n_splits=4
    )

    fold_rmse = []

    for fold, (train_idx, valid_idx) in enumerate(
        tscv.split(X),
        start=1
    ):

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        pipeline = build_rf_pipeline(
            feature_list
        )

        pipeline.fit(
            X_train,
            y_train
        )

        predictions = pipeline.predict(
            X_valid
        )

        y_valid_np = y_valid.to_numpy()

        valid_mask = (
            np.isfinite(y_valid_np)
            &
            np.isfinite(predictions)
        )

        if valid_mask.sum() == 0:
            continue

        rmse = np.sqrt(
            mean_squared_error(
                y_valid_np[valid_mask],
                predictions[valid_mask]
            )
        )

        fold_rmse.append(
            float(rmse)
        )

        print(
            f"    Fold {fold}: RMSE = {rmse:.4f}"
        )

    if not fold_rmse:

        return {
            "Feature Set": feature_set_name,
            "Features": len(feature_list),
            "Mean RMSE": np.nan,
            "Std RMSE": np.nan,
            "Folds": 0
        }

    return {
        "Feature Set": feature_set_name,
        "Features": len(feature_list),
        "Mean RMSE": float(
            np.mean(fold_rmse)
        ),
        "Std RMSE": float(
            np.std(fold_rmse)
        ),
        "Folds": len(fold_rmse)
    }


# ------------------------------------------------------------
# 9. TARGETED FEATURE COMBINATIONS FOR STEP 14.9
# ------------------------------------------------------------

targeted_sets = {

    "Current + Lag":
        current_features
        + lag_features
        + station_features,

    "Current + Rolling":
        current_features
        + rolling_features
        + station_features,

    "Current + Lag + Rolling":
        current_features
        + lag_features
        + rolling_features
        + station_features,

    "Current + Lag + Rolling + Calendar":
        current_features
        + lag_features
        + rolling_features
        + calendar_features
        + station_features,

    "AQI + Lag + Rolling":
        ["AQI"]
        + lag_features
        + rolling_features
        + station_features
}


# ------------------------------------------------------------
# 10. RESTORE CHECK
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 14.9 ENVIRONMENT READY")
print("=" * 70)

print("Dataset shape       :", df14.shape)
print("Stations            :", df14["StationId"].nunique())
print("Current features    :", len(current_features))
print("Lag features        :", len(lag_features))
print("Rolling features    :", len(rolling_features))
print("Calendar features   :", len(calendar_features))
print("Event features      :", len(event_features))
print("Station features    :", len(station_features))
print("Total model features:", len(model_features))
print("Target              :", TARGET)
print("RF estimators       :", rf_params.get("n_estimators"))
print("TimeSeries folds    : 4")

print("\nFeature names verified successfully.")
print("evaluate_feature_set() is ready.")
print("targeted_sets is ready.")

print("=" * 70)

In [ ]:
# ============================================================
# PHASE 14.9 — TARGETED FEATURE-SET ABLATION
# ============================================================

import os
import json
import pandas as pd

# Always initialize after a runtime reset
ablation_candidates = []

print("=" * 70)
print("PHASE 14.9 — TARGETED FEATURE-SET EVALUATION")
print("=" * 70)

for name, feature_list in targeted_sets.items():

    print(f"\nEvaluating: {name}")
    print(f"Number of features: {len(set(feature_list))}")

    result = evaluate_feature_set(
        list(dict.fromkeys(feature_list)),
        name
    )

    ablation_candidates.append(result)

# ------------------------------------------------------------
# Build leaderboard
# ------------------------------------------------------------

ablation_results = pd.DataFrame(
    ablation_candidates
)

ablation_results = ablation_results.sort_values(
    by="Mean RMSE",
    ascending=True
).reset_index(drop=True)

print("\n" + "=" * 70)
print("PHASE 14.9 — RESULTS")
print("=" * 70)

display(ablation_results)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

RESULTS_DIR = (
    "/content/drive/MyDrive/Delhi-AQI-Prediction/"
    "outputs/reports"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "phase14_9_targeted_ablation_results.csv"
)

ablation_results.to_csv(
    RESULTS_PATH,
    index=False
)

print("\nSaved:", RESULTS_PATH)

print("=" * 70)
print("PHASE 14.9 COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# STEP 0 — RECONNECT GOOGLE DRIVE IN NEW COLAB RUNTIME
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

print("=" * 70)
print("GOOGLE DRIVE CONNECTED")
print("=" * 70)
print("Project root:")
print(PROJECT_ROOT)
print("=" * 70)

In [ ]:
# ============================================================
# STEP 1 — CHECK SAVED PHASE 14 FILES
# ============================================================

import os

REPORT_DIR = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports"
)

print("Report directory:", REPORT_DIR)
print("\nPhase 14 files found:\n")

for filename in sorted(os.listdir(REPORT_DIR)):
    if "phase14" in filename.lower():
        print("✓", filename)

In [ ]:
# ============================================================
# RESTORE PHASE 14.9 RESULTS FROM GOOGLE DRIVE
# ============================================================

import os
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_9_targeted_ablation_results.csv"
)

if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(
        f"Saved Phase 14.9 results not found:\n{RESULTS_PATH}"
    )

ablation_results = pd.read_csv(RESULTS_PATH)

print("=" * 70)
print("PHASE 14.9 RESULTS RESTORED")
print("=" * 70)
print("File:", RESULTS_PATH)
print("\nColumns:")
print(ablation_results.columns.tolist())

print("\nSaved results:")
display(ablation_results)

print("=" * 70)

In [ ]:
# ============================================================
# PHASE 14.10 — BUILD ABLATION LEADERBOARD
# ============================================================

import os
import numpy as np
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

LEADERBOARD_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_10_ablation_leaderboard.csv"
)

# Confirm restored data exists
if "ablation_results" not in globals():
    raise RuntimeError(
        "ablation_results is not loaded. "
        "Run the Phase 14.9 restore cell first."
    )

# Use the actual Phase 14.9 metric column
if "Mean RMSE" not in ablation_results.columns:
    raise ValueError(
        "Expected 'Mean RMSE' column was not found."
    )

# Sort by lowest RMSE
ablation_results = (
    ablation_results
    .sort_values(
        by="Mean RMSE",
        ascending=True
    )
    .reset_index(drop=True)
)

# Add rank
ablation_results.insert(
    0,
    "Rank",
    np.arange(1, len(ablation_results) + 1)
)

print("=" * 70)
print("PHASE 14.10 — ABLATION LEADERBOARD")
print("=" * 70)

display(
    ablation_results[
        [
            "Rank",
            "Feature Set",
            "Features",
            "Mean RMSE",
            "Std RMSE",
            "Folds"
        ]
    ]
)

# Save leaderboard
ablation_results.to_csv(
    LEADERBOARD_PATH,
    index=False
)

print("\nSaved:")
print(LEADERBOARD_PATH)

print("\n" + "=" * 70)
print("PHASE 14.10 COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 14.11 — ABLATION CANDIDATE vs CURRENT CHAMPION
# ============================================================

import os
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

LEADERBOARD_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_10_ablation_leaderboard.csv"
)

if not os.path.exists(LEADERBOARD_PATH):
    raise FileNotFoundError(
        "Phase 14.10 leaderboard not found:\n"
        + LEADERBOARD_PATH
    )

# ------------------------------------------------------------
# Load Phase 14.10 leaderboard
# ------------------------------------------------------------

leaderboard = pd.read_csv(
    LEADERBOARD_PATH
)

# ------------------------------------------------------------
# Identify the best Phase 14.9/14.10 candidate
# ------------------------------------------------------------

best_row = (
    leaderboard
    .sort_values(
        by="Mean RMSE",
        ascending=True
    )
    .iloc[0]
)

candidate_name = best_row["Feature Set"]
candidate_rmse = float(best_row["Mean RMSE"])

# ------------------------------------------------------------
# Current Phase 13B champion benchmark
# ------------------------------------------------------------

CHAMPION_NAME = "Tuned Random Forest"
CHAMPION_RMSE = 8.386059444844367
CHAMPION_ACCURACY_10 = 93.77083219496203

# ------------------------------------------------------------
# Comparison
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Configuration": [
        CHAMPION_NAME,
        f"Phase 14 Candidate — {candidate_name}"
    ],
    "RMSE": [
        CHAMPION_RMSE,
        candidate_rmse
    ],
    "Accuracy ±10 AQI": [
        CHAMPION_ACCURACY_10,
        None
    ]
})

print("=" * 70)
print("PHASE 14.11 — CANDIDATE vs CURRENT CHAMPION")
print("=" * 70)

display(comparison)

print("\nCurrent champion RMSE :", round(CHAMPION_RMSE, 6))
print("Candidate RMSE       :", round(candidate_rmse, 6))
print(
    "RMSE difference      :",
    round(candidate_rmse - CHAMPION_RMSE, 6)
)

print("\nCandidate feature set :", candidate_name)
print("Candidate features    :", int(best_row["Features"]))

print("\nNOTE:")
print(
    "The Phase 14 candidate RMSE is from TimeSeriesSplit "
    "ablation validation. The 8.386059 champion value is the "
    "Phase 13B benchmark. This comparison is for reference; "
    "final promotion requires evaluation on the same untouched "
    "future test period."
)

# ------------------------------------------------------------
# Save comparison
# ------------------------------------------------------------

SAVE_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_11_candidate_champion_comparison.csv"
)

comparison.to_csv(
    SAVE_PATH,
    index=False
)

print("\nSaved:", SAVE_PATH)

print("=" * 70)
print("PHASE 14.11 COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 14.12 — FREEZE BEST FEATURE CANDIDATE
# ============================================================

import os
import json
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "delhi_10_station_feature_engineered.parquet"
)

LEADERBOARD_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_10_ablation_leaderboard.csv"
)

SAVE_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_12_candidate_feature_config.json"
)

# ------------------------------------------------------------
# Load the actual saved dataset and leaderboard
# ------------------------------------------------------------

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found:\n{DATA_PATH}")

if not os.path.exists(LEADERBOARD_PATH):
    raise FileNotFoundError(
        f"Phase 14.10 leaderboard not found:\n{LEADERBOARD_PATH}"
    )

df = pd.read_parquet(DATA_PATH)

leaderboard = pd.read_csv(
    LEADERBOARD_PATH
)

# ------------------------------------------------------------
# Identify the best Phase 14 candidate
# ------------------------------------------------------------

best_row = (
    leaderboard
    .sort_values(
        by="Mean RMSE",
        ascending=True
    )
    .iloc[0]
)

candidate_name = best_row["Feature Set"]

# ------------------------------------------------------------
# Reconstruct feature groups using ACTUAL column names
# ------------------------------------------------------------

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

rolling_features = [
    "AQI_rolling_mean_3h",
    "AQI_rolling_std_3h",
    "AQI_rolling_mean_12h",
    "AQI_rolling_std_12h",

    "PM2.5_rolling_mean_3h",
    "PM2.5_rolling_std_3h",
    "PM2.5_rolling_mean_12h",
    "PM2.5_rolling_std_12h",

    "PM10_rolling_mean_3h",
    "PM10_rolling_std_3h",
    "PM10_rolling_mean_12h",
    "PM10_rolling_std_12h",

    "NO2_rolling_mean_3h",
    "NO2_rolling_std_3h",
    "NO2_rolling_mean_12h",
    "NO2_rolling_std_12h"
]

station_features = [
    "StationId"
]

# ------------------------------------------------------------
# Build candidate configuration
# ------------------------------------------------------------

if candidate_name == "Current + Rolling":

    candidate_features = (
        current_features
        + rolling_features
        + station_features
    )

else:
    raise ValueError(
        f"Unexpected candidate feature set: {candidate_name}"
    )

candidate_features = list(
    dict.fromkeys(candidate_features)
)

# ------------------------------------------------------------
# Verify every candidate feature exists
# ------------------------------------------------------------

missing = [
    feature
    for feature in candidate_features
    if feature not in df.columns
]

if missing:
    raise ValueError(
        "Candidate features missing from dataset:\n"
        + "\n".join(missing)
    )

# ------------------------------------------------------------
# Save configuration
# ------------------------------------------------------------

candidate_config = {
    "phase": "14",
    "step": "14.12",

    "candidate_name": candidate_name,

    "selection_basis": (
        "Lowest Mean RMSE in Phase 14.10 "
        "targeted feature-set ablation"
    ),

    "validation_rmse": float(
        best_row["Mean RMSE"]
    ),

    "validation_rmse_std": float(
        best_row["Std RMSE"]
    ),

    "feature_count": len(candidate_features),

    "feature_groups": {
        "Current": current_features,
        "Rolling": rolling_features,
        "Station": station_features
    },

    "features": candidate_features,

    "target": "AQI_t_plus_1",

    "stations": int(
        df["StationId"].nunique()
    ),

    "status": "CANDIDATE — NOT YET PROMOTED"
}

with open(SAVE_PATH, "w") as f:
    json.dump(
        candidate_config,
        f,
        indent=4
    )

# ------------------------------------------------------------
# Display confirmation
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 14.12 — CANDIDATE FEATURE CONFIGURATION SAVED")
print("=" * 70)

print("Candidate          :", candidate_name)
print("Feature count      :", len(candidate_features))
print("Validation RMSE    :", round(
    float(best_row["Mean RMSE"]), 6
))
print("Target             :", candidate_config["target"])
print("Stations           :", candidate_config["stations"])
print("Status             :", candidate_config["status"])

print("\nFeature groups:")
for group, features in candidate_config["feature_groups"].items():
    print(f"  {group:10s}: {len(features)}")

print("\nSaved:")
print(SAVE_PATH)

print("=" * 70)
print("PHASE 14.12 COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 14.13 — CORRECTED FEATURE IMPORTANCE ANALYSIS
# Handles StationId as a categorical feature
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "delhi_10_station_feature_engineered.parquet"
)

CONFIG_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_12_candidate_feature_config.json"
)

SAVE_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_13_feature_importance.csv"
)

GROUP_SAVE_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_13_feature_group_importance.csv"
)


# ------------------------------------------------------------
# 2. VERIFY FILES
# ------------------------------------------------------------

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found:\n{DATA_PATH}"
    )

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(
        f"Phase 14.12 configuration not found:\n{CONFIG_PATH}"
    )


# ------------------------------------------------------------
# 3. LOAD DATA + CANDIDATE CONFIG
# ------------------------------------------------------------

df = pd.read_parquet(DATA_PATH)

df["Datetime"] = pd.to_datetime(
    df["Datetime"],
    errors="coerce"
)

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

candidate_features = config["features"]
TARGET = config["target"]


# ------------------------------------------------------------
# 4. VERIFY FEATURES
# ------------------------------------------------------------

missing_features = [
    feature
    for feature in candidate_features
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        "Missing candidate features:\n"
        + "\n".join(missing_features)
    )


print("=" * 70)
print("PHASE 14.13 — FEATURE IMPORTANCE ANALYSIS")
print("=" * 70)

print("Candidate feature set :", config["candidate_name"])
print("Number of features   :", len(candidate_features))
print("Target               :", TARGET)


# ------------------------------------------------------------
# 5. PREPARE DATA
# ------------------------------------------------------------

work = df[
    ["Datetime"] + candidate_features + [TARGET]
].copy()

work = work.replace(
    [np.inf, -np.inf],
    np.nan
)

work[TARGET] = pd.to_numeric(
    work[TARGET],
    errors="coerce"
)

work = work.dropna(
    subset=["Datetime", TARGET]
)

work = work.sort_values(
    "Datetime"
).reset_index(drop=True)


# ------------------------------------------------------------
# 6. HISTORICAL TRAINING PORTION
# ------------------------------------------------------------

split_index = int(
    len(work) * 0.80
)

train = work.iloc[
    :split_index
].copy()

X_train = train[candidate_features].copy()
y_train = train[TARGET].copy()


print("\nTotal valid rows :", len(work))
print("Training rows    :", len(train))
print(
    "Training period  :",
    train["Datetime"].min(),
    "to",
    train["Datetime"].max()
)


# ------------------------------------------------------------
# 7. IDENTIFY NUMERIC + CATEGORICAL FEATURES
# ------------------------------------------------------------

categorical_features = []

if "StationId" in candidate_features:
    categorical_features.append("StationId")

numeric_features = [
    feature
    for feature in candidate_features
    if feature not in categorical_features
]


print("\nNumeric features    :", len(numeric_features))
print("Categorical features:", len(categorical_features))

if categorical_features:
    print(
        "Categorical column  :",
        categorical_features
    )


# ------------------------------------------------------------
# 8. BUILD PREPROCESSING PIPELINE
# ------------------------------------------------------------

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

transformers = [
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    )
]

if categorical_features:
    transformers.append(
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop"
)


# ------------------------------------------------------------
# 9. TRANSFORM TRAINING DATA
# ------------------------------------------------------------

X_train_transformed = preprocessor.fit_transform(
    X_train,
    y_train
)

transformed_feature_names = (
    preprocessor.get_feature_names_out()
)

print(
    "\nOriginal candidate features :",
    len(candidate_features)
)

print(
    "Transformed model features  :",
    len(transformed_feature_names)
)


# ------------------------------------------------------------
# 10. TRAIN RANDOM FOREST
# ------------------------------------------------------------

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train_transformed,
    y_train
)


# ------------------------------------------------------------
# 11. RAW FEATURE IMPORTANCE
# ------------------------------------------------------------

raw_importance = pd.DataFrame({
    "Transformed Feature":
        transformed_feature_names,
    "Importance":
        rf.feature_importances_
})


# ------------------------------------------------------------
# 12. MAP TRANSFORMED FEATURES BACK TO ORIGINAL FEATURES
# ------------------------------------------------------------

def map_to_original_feature(name):

    # Numeric transformed names:
    # numeric__AQI
    if name.startswith("numeric__"):
        return name.replace(
            "numeric__",
            "",
            1
        )

    # Categorical transformed names:
    # categorical__StationId_DL002
    if name.startswith("categorical__StationId_"):
        return "StationId"

    return name


raw_importance["Feature"] = (
    raw_importance["Transformed Feature"]
    .apply(map_to_original_feature)
)


# ------------------------------------------------------------
# 13. AGGREGATE STATION ONE-HOT IMPORTANCES
# ------------------------------------------------------------

importance = (
    raw_importance
    .groupby("Feature", as_index=False)["Importance"]
    .sum()
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

importance["Importance_%"] = (
    importance["Importance"] * 100
)

importance["Cumulative_%"] = (
    importance["Importance_%"]
    .cumsum()
)


# ------------------------------------------------------------
# 14. DISPLAY FEATURE IMPORTANCE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE IMPORTANCE — TOP FEATURES")
print("=" * 70)

display(
    importance
)


# ------------------------------------------------------------
# 15. FEATURE GROUP MAPPING
# ------------------------------------------------------------

def get_feature_group(feature):

    if feature in config["feature_groups"]["Current"]:
        return "Current"

    if feature in config["feature_groups"]["Rolling"]:
        return "Rolling"

    if feature in config["feature_groups"]["Station"]:
        return "Station"

    return "Other"


importance["Feature Group"] = (
    importance["Feature"]
    .apply(get_feature_group)
)


# ------------------------------------------------------------
# 16. GROUP IMPORTANCE
# ------------------------------------------------------------

group_summary = (
    importance
    .groupby("Feature Group", as_index=False)["Importance"]
    .sum()
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

group_summary["Importance_%"] = (
    group_summary["Importance"] * 100
)

group_summary["Cumulative_%"] = (
    group_summary["Importance_%"]
    .cumsum()
)


print("\n" + "=" * 70)
print("FEATURE GROUP IMPORTANCE")
print("=" * 70)

display(
    group_summary
)


# ------------------------------------------------------------
# 17. SAVE FEATURE IMPORTANCE
# ------------------------------------------------------------

importance.to_csv(
    SAVE_PATH,
    index=False
)

group_summary.to_csv(
    GROUP_SAVE_PATH,
    index=False
)


# ------------------------------------------------------------
# 18. FINAL CONFIRMATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PHASE 14.13 COMPLETED SUCCESSFULLY")
print("=" * 70)

print("Feature importance saved:")
print(SAVE_PATH)

print("\nFeature group importance saved:")
print(GROUP_SAVE_PATH)

print("\nNo features were removed.")
print("This step is analysis only.")
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 14.14 — CONTROLLED FEATURE PRUNING CANDIDATES
# ============================================================

import os
import json
import pandas as pd

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

IMPORTANCE_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_13_feature_importance.csv"
)

CONFIG_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_12_candidate_feature_config.json"
)

SAVE_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_14_pruning_candidates.json"
)

# ------------------------------------------------------------
# 2. LOAD SAVED RESULTS
# ------------------------------------------------------------

if not os.path.exists(IMPORTANCE_PATH):
    raise FileNotFoundError(
        f"Feature importance file not found:\n{IMPORTANCE_PATH}"
    )

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(
        f"Phase 14.12 configuration not found:\n{CONFIG_PATH}"
    )

importance = pd.read_csv(
    IMPORTANCE_PATH
)

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

full_features = list(
    config["features"]
)

# ------------------------------------------------------------
# 3. BASIC VALIDATION
# ------------------------------------------------------------

if "Feature" not in importance.columns:
    raise ValueError(
        "Expected 'Feature' column not found."
    )

if "Importance" not in importance.columns:
    raise ValueError(
        "Expected 'Importance' column not found."
    )

importance = (
    importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. BUILD CONTROLLED PRUNING CANDIDATES
# ------------------------------------------------------------

pruning_candidates = {}

# Candidate A — Full feature set
pruning_candidates[
    "Full 29 Features"
] = full_features.copy()

# Candidate B — Importance >= 0.00030
features_0030 = (
    importance[
        importance["Importance"] >= 0.00030
    ]["Feature"]
    .tolist()
)

pruning_candidates[
    "Importance >= 0.00030"
] = list(
    dict.fromkeys(features_0030)
)

# Candidate C — Importance >= 0.00050
features_0050 = (
    importance[
        importance["Importance"] >= 0.00050
    ]["Feature"]
    .tolist()
)

pruning_candidates[
    "Importance >= 0.00050"
] = list(
    dict.fromkeys(features_0050)
)

# Candidate D — AQI + all rolling features
rolling_features = [
    feature
    for feature in full_features
    if "_rolling_" in feature
]

pruning_candidates[
    "AQI + Rolling + Station"
] = list(
    dict.fromkeys(
        ["AQI"]
        + rolling_features
        + (
            ["StationId"]
            if "StationId" in full_features
            else []
        )
    )
)

# ------------------------------------------------------------
# 5. VERIFY ALL CANDIDATES
# ------------------------------------------------------------

for name, features in pruning_candidates.items():

    if "AQI" not in features:
        raise ValueError(
            f"{name} does not contain AQI."
        )

    missing_from_full = [
        feature
        for feature in features
        if feature not in full_features
    ]

    if missing_from_full:
        raise ValueError(
            f"{name} contains unexpected features:\n"
            + "\n".join(missing_from_full)
        )

# ------------------------------------------------------------
# 6. DISPLAY CANDIDATES
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 14.14 — CONTROLLED PRUNING CANDIDATES")
print("=" * 70)

summary = []

for name, features in pruning_candidates.items():

    summary.append({
        "Candidate": name,
        "Feature Count": len(features)
    })

    print(
        f"{name:30s} : {len(features)} features"
    )

summary_df = pd.DataFrame(summary)

print("\nCandidate summary:")
display(summary_df)

# ------------------------------------------------------------
# 7. SAVE CANDIDATES
# ------------------------------------------------------------

save_object = {
    "source_step": "14.13",
    "base_candidate": config["candidate_name"],
    "base_feature_count": len(full_features),
    "candidates": pruning_candidates,
    "status": "CANDIDATES ONLY — NOT EVALUATED YET"
}

with open(SAVE_PATH, "w") as f:
    json.dump(
        save_object,
        f,
        indent=4
    )

print("\nSaved:")
print(SAVE_PATH)

print("=" * 70)
print("PHASE 14.14 COMPLETED")
print("=" * 70)
print("No features were removed from the project dataset.")
print("Candidate subsets were created for controlled testing.")
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 14.15 — CONTROLLED FEATURE PRUNING EVALUATION
#
# Resume-safe:
# - Loads candidates from Phase 14.14
# - Saves results after EVERY completed candidate
# - Skips candidates already completed if runtime reconnects
#
# Primary metric: Mean RMSE
# Secondary metrics: Mean MAE, Accuracy ±10 AQI
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "delhi_10_station_feature_engineered.parquet"
)

CANDIDATES_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_14_pruning_candidates.json"
)

RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_15_pruning_results.csv"
)

# ------------------------------------------------------------
# 2. VERIFY FILES
# ------------------------------------------------------------

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found:\n{DATA_PATH}"
    )

if not os.path.exists(CANDIDATES_PATH):
    raise FileNotFoundError(
        f"Phase 14.14 candidates not found:\n{CANDIDATES_PATH}"
    )

# ------------------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------------------

df = pd.read_parquet(DATA_PATH)

df["Datetime"] = pd.to_datetime(
    df["Datetime"],
    errors="coerce"
)

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = (
    df
    .dropna(
        subset=[
            "Datetime",
            "AQI_t_plus_1"
        ]
    )
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

TARGET = "AQI_t_plus_1"

# ------------------------------------------------------------
# 4. LOAD PRUNING CANDIDATES
# ------------------------------------------------------------

with open(CANDIDATES_PATH, "r") as f:
    candidate_config = json.load(f)

pruning_candidates = candidate_config["candidates"]

# ------------------------------------------------------------
# 5. LOAD PREVIOUS RESULTS IF THEY EXIST
# ------------------------------------------------------------

if os.path.exists(RESULTS_PATH):

    results_df = pd.read_csv(
        RESULTS_PATH
    )

    completed_candidates = set(
        results_df["Candidate"].astype(str)
    )

    print(
        "Existing results found:",
        len(results_df)
    )

else:

    results_df = pd.DataFrame()

    completed_candidates = set()

# ------------------------------------------------------------
# 6. RANDOM FOREST CONFIGURATION
#
# Same RF configuration used for Phase 14.13
# ------------------------------------------------------------

RF_PARAMS = {
    "n_estimators": 300,
    "random_state": 42,
    "n_jobs": -1
}

# ------------------------------------------------------------
# 7. BUILD LEAKAGE-SAFE PIPELINE
# ------------------------------------------------------------

def build_pipeline(feature_list):

    categorical_features = []
    numeric_features = []

    for feature in feature_list:

        if (
            df[feature].dtype == "object"
            or str(df[feature].dtype).startswith("category")
        ):
            categorical_features.append(feature)

        else:
            numeric_features.append(feature)

    transformers = []

    # Numeric preprocessing
    if numeric_features:

        numeric_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            )
        ])

        transformers.append(
            (
                "numeric",
                numeric_pipeline,
                numeric_features
            )
        )

    # Categorical preprocessing
    if categorical_features:

        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ])

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    model = RandomForestRegressor(
        **RF_PARAMS
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            model
        )
    ])


# ------------------------------------------------------------
# 8. EVALUATE ONE CANDIDATE
# ------------------------------------------------------------

def evaluate_candidate(
    candidate_name,
    feature_list
):

    feature_list = list(
        dict.fromkeys(feature_list)
    )

    # Verify features exist
    missing = [
        feature
        for feature in feature_list
        if feature not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing features for {candidate_name}:\n"
            + "\n".join(missing)
        )

    work = df[
        ["Datetime", TARGET] + feature_list
    ].copy()

    work = work.replace(
        [np.inf, -np.inf],
        np.nan
    )

    work[TARGET] = pd.to_numeric(
        work[TARGET],
        errors="coerce"
    )

    work = (
        work
        .dropna(
            subset=[
                "Datetime",
                TARGET
            ]
        )
        .sort_values("Datetime")
        .reset_index(drop=True)
    )

    X = work[feature_list]
    y = work[TARGET]

    tscv = TimeSeriesSplit(
        n_splits=4
    )

    fold_rmse = []
    fold_mae = []
    fold_accuracy_10 = []

    for fold, (
        train_idx,
        valid_idx
    ) in enumerate(
        tscv.split(X),
        start=1
    ):

        print(
            f"    Fold {fold}/4 ..."
        )

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        pipeline = build_pipeline(
            feature_list
        )

        pipeline.fit(
            X_train,
            y_train
        )

        predictions = pipeline.predict(
            X_valid
        )

        y_valid_np = y_valid.to_numpy()
        predictions_np = np.asarray(
            predictions
        )

        valid_mask = (
            np.isfinite(y_valid_np)
            &
            np.isfinite(predictions_np)
        )

        if valid_mask.sum() == 0:
            raise ValueError(
                f"No valid predictions in fold {fold}"
            )

        actual = y_valid_np[
            valid_mask
        ]

        predicted = predictions_np[
            valid_mask
        ]

        rmse = np.sqrt(
            mean_squared_error(
                actual,
                predicted
            )
        )

        mae = mean_absolute_error(
            actual,
            predicted
        )

        accuracy_10 = (
            np.mean(
                np.abs(
                    actual - predicted
                ) <= 10
            ) * 100
        )

        fold_rmse.append(
            float(rmse)
        )

        fold_mae.append(
            float(mae)
        )

        fold_accuracy_10.append(
            float(accuracy_10)
        )

        print(
            f"      RMSE = {rmse:.4f} | "
            f"MAE = {mae:.4f} | "
            f"±10 = {accuracy_10:.2f}%"
        )

    return {
        "Candidate": candidate_name,
        "Feature Count": len(feature_list),

        "Mean RMSE": float(
            np.mean(fold_rmse)
        ),

        "Std RMSE": float(
            np.std(fold_rmse)
        ),

        "Mean MAE": float(
            np.mean(fold_mae)
        ),

        "Std MAE": float(
            np.std(fold_mae)
        ),

        "Mean Accuracy ±10": float(
            np.mean(fold_accuracy_10)
        ),

        "Std Accuracy ±10": float(
            np.std(fold_accuracy_10)
        ),

        "Fold1 RMSE": fold_rmse[0],
        "Fold2 RMSE": fold_rmse[1],
        "Fold3 RMSE": fold_rmse[2],
        "Fold4 RMSE": fold_rmse[3],

        "Folds": 4
    }


# ------------------------------------------------------------
# 9. EVALUATE CANDIDATES
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 14.15 — CONTROLLED PRUNING EVALUATION")
print("=" * 70)

print(
    "Dataset rows:",
    len(df)
)

print(
    "Random Forest estimators:",
    RF_PARAMS["n_estimators"]
)

print(
    "Candidates:",
    len(pruning_candidates)
)

print("=" * 70)


for candidate_name, feature_list in pruning_candidates.items():

    # Resume-safe behavior
    if candidate_name in completed_candidates:

        print(
            f"\nSKIPPING already completed: "
            f"{candidate_name}"
        )

        continue

    print("\n" + "=" * 60)
    print(
        f"EVALUATING: {candidate_name}"
    )
    print(
        f"Feature count: {len(feature_list)}"
    )
    print("=" * 60)

    result = evaluate_candidate(
        candidate_name,
        feature_list
    )

    # Add result to dataframe
    new_result_df = pd.DataFrame(
        [result]
    )

    results_df = pd.concat(
        [
            results_df,
            new_result_df
        ],
        ignore_index=True
    )

    # Save IMMEDIATELY after candidate
    results_df.to_csv(
        RESULTS_PATH,
        index=False
    )

    completed_candidates.add(
        candidate_name
    )

    print(
        f"\n✓ Saved completed candidate: "
        f"{candidate_name}"
    )

    print(
        f"  Mean RMSE: "
        f"{result['Mean RMSE']:.6f}"
    )

    print(
        f"  Mean MAE: "
        f"{result['Mean MAE']:.6f}"
    )

    print(
        f"  Mean ±10 Accuracy: "
        f"{result['Mean Accuracy ±10']:.2f}%"
    )


# ------------------------------------------------------------
# 10. BUILD FINAL LEADERBOARD
# ------------------------------------------------------------

if len(results_df) > 0:

    results_df = (
        results_df
        .sort_values(
            by="Mean RMSE",
            ascending=True
        )
        .reset_index(drop=True)
    )

    results_df["Rank"] = (
        np.arange(
            1,
            len(results_df) + 1
        )
    )

    # Put Rank first
    rank_col = results_df.pop(
        "Rank"
    )

    results_df.insert(
        0,
        "Rank",
        rank_col
    )

    results_df.to_csv(
        RESULTS_PATH,
        index=False
    )

    print("\n" + "=" * 70)
    print("PHASE 14.15 — CURRENT LEADERBOARD")
    print("=" * 70)

    display(
        results_df[
            [
                "Rank",
                "Candidate",
                "Feature Count",
                "Mean RMSE",
                "Std RMSE",
                "Mean MAE",
                "Mean Accuracy ±10",
                "Folds"
            ]
        ]
    )


print("\n" + "=" * 70)
print("PHASE 14.15 EVALUATION FINISHED")
print("=" * 70)

print(
    "Results saved to:"
)
print(RESULTS_PATH)

print("=" * 70)

In [ ]:
# ============================================================
# PHASE 14.16 — FREEZE BEST PHASE 14 FEATURE CANDIDATE
# ============================================================

import os
import json
import pandas as pd

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_15_pruning_results.csv"
)

CANDIDATES_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_14_pruning_candidates.json"
)

IMPORTANCE_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_13_feature_importance.csv"
)

SAVE_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_16_final_candidate_config.json"
)

# ------------------------------------------------------------
# 2. VERIFY FILES
# ------------------------------------------------------------

for path in [
    RESULTS_PATH,
    CANDIDATES_PATH,
    IMPORTANCE_PATH
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )

# ------------------------------------------------------------
# 3. LOAD RESULTS
# ------------------------------------------------------------

results = pd.read_csv(
    RESULTS_PATH
)

with open(CANDIDATES_PATH, "r") as f:
    candidate_data = json.load(f)

# ------------------------------------------------------------
# 4. SELECT BEST CANDIDATE
# ------------------------------------------------------------

best_row = (
    results
    .sort_values(
        by="Mean RMSE",
        ascending=True
    )
    .iloc[0]
)

best_name = str(
    best_row["Candidate"]
)

best_rmse = float(
    best_row["Mean RMSE"]
)

best_mae = float(
    best_row["Mean MAE"]
)

best_accuracy_10 = float(
    best_row["Mean Accuracy ±10"]
)

# ------------------------------------------------------------
# 5. RETRIEVE ACTUAL FEATURE LIST
# ------------------------------------------------------------

candidate_dict = candidate_data["candidates"]

if best_name not in candidate_dict:
    raise ValueError(
        f"Feature list for '{best_name}' "
        "was not found in Phase 14.14 candidates."
    )

best_features = list(
    dict.fromkeys(
        candidate_dict[best_name]
    )
)

# ------------------------------------------------------------
# 6. VERIFY FEATURE COUNT
# ------------------------------------------------------------

reported_count = int(
    best_row["Feature Count"]
)

actual_count = len(
    best_features
)

if reported_count != actual_count:
    raise ValueError(
        f"Feature count mismatch: "
        f"results={reported_count}, "
        f"actual={actual_count}"
    )

# ------------------------------------------------------------
# 7. LOAD IMPORTANCE FOR DOCUMENTATION
# ------------------------------------------------------------

importance = pd.read_csv(
    IMPORTANCE_PATH
)

importance_lookup = (
    importance[
        importance["Feature"].isin(
            best_features
        )
    ][
        [
            "Feature",
            "Importance",
            "Importance_%"
        ]
    ]
    .sort_values(
        "Importance",
        ascending=False
    )
    .to_dict(
        orient="records"
    )
)

# ------------------------------------------------------------
# 8. SAVE FROZEN CANDIDATE CONFIGURATION
# ------------------------------------------------------------

config = {
    "phase": "14",
    "step": "14.16",

    "candidate_name": best_name,

    "feature_count": actual_count,

    "features": best_features,

    "validation_metrics": {
        "mean_rmse": best_rmse,
        "mean_mae": best_mae,
        "mean_accuracy_plus_minus_10": best_accuracy_10,
        "folds": int(best_row["Folds"])
    },

    "selection_basis": (
        "Lowest Mean RMSE among controlled "
        "feature-pruning candidates in Phase 14.15"
    ),

    "feature_importance": importance_lookup,

    "target": "AQI_t_plus_1",

    "current_project_champion": {
        "model": "Tuned Random Forest",
        "rmse": 8.386059444844367,
        "accuracy_plus_minus_10": 93.77083219496203
    },

    "status": (
        "FROZEN CANDIDATE — "
        "NOT YET PROMOTED TO FINAL MODEL"
    )
}

with open(SAVE_PATH, "w") as f:
    json.dump(
        config,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 9. DISPLAY
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 14.16 — FROZEN CANDIDATE")
print("=" * 70)

print("Candidate           :", best_name)
print("Feature count       :", actual_count)
print("Mean RMSE           :", round(best_rmse, 6))
print("Mean MAE            :", round(best_mae, 6))
print(
    "Mean Accuracy ±10   :",
    round(best_accuracy_10, 2),
    "%"
)

print("\nFeatures:")
for i, feature in enumerate(
    best_features,
    start=1
):
    print(f"{i:2d}. {feature}")

print("\nStatus:")
print(config["status"])

print("\nSaved:")
print(SAVE_PATH)

print("=" * 70)
print("PHASE 14.16 COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 14.17 — FINAL FUTURE-TEST COMPARISON
#
# Compare:
#   A) Current Tuned Random Forest
#   B) Phase 14 frozen 7-feature candidate
#
# SAME chronological 80/20 split
# SAME Random Forest configuration
# SAME untouched future test period
#
# Metrics:
#   MAE
#   RMSE
#   R²
#   Accuracy ±10 AQI
#   Accuracy ±20 AQI
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "delhi_10_station_feature_engineered.parquet"
)

CANDIDATE_CONFIG_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_16_final_candidate_config.json"
)

TUNED_RF_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "phase13B_tuned_random_forest.joblib"
)

SAVE_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_17_future_test_comparison.csv"
)

DETAIL_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_17_future_test_details.json"
)


# ------------------------------------------------------------
# 2. VERIFY REQUIRED FILES
# ------------------------------------------------------------

for path in [
    DATA_PATH,
    CANDIDATE_CONFIG_PATH,
    TUNED_RF_PATH
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )


# ------------------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------------------

df = pd.read_parquet(
    DATA_PATH
)

df["Datetime"] = pd.to_datetime(
    df["Datetime"],
    errors="coerce"
)

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = (
    df
    .dropna(
        subset=[
            "Datetime",
            "AQI_t_plus_1"
        ]
    )
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

TARGET = "AQI_t_plus_1"


# ------------------------------------------------------------
# 4. LOAD FROZEN PHASE 14.16 CANDIDATE
# ------------------------------------------------------------

with open(
    CANDIDATE_CONFIG_PATH,
    "r"
) as f:
    candidate_config = json.load(f)

candidate_features = list(
    candidate_config["features"]
)

candidate_name = candidate_config[
    "candidate_name"
]


# ------------------------------------------------------------
# 5. DEFINE CURRENT FULL FEATURE SET
#    Using ACTUAL saved column names
# ------------------------------------------------------------

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    f"{feature}_lag_{lag}h"
    for lag in [1, 2, 12]
    for feature in current_features
]

rolling_features = [
    "AQI_rolling_mean_3h",
    "AQI_rolling_std_3h",
    "AQI_rolling_mean_12h",
    "AQI_rolling_std_12h",

    "PM2.5_rolling_mean_3h",
    "PM2.5_rolling_std_3h",
    "PM2.5_rolling_mean_12h",
    "PM2.5_rolling_std_12h",

    "PM10_rolling_mean_3h",
    "PM10_rolling_std_3h",
    "PM10_rolling_mean_12h",
    "PM10_rolling_std_12h",

    "NO2_rolling_mean_3h",
    "NO2_rolling_std_3h",
    "NO2_rolling_mean_12h",
    "NO2_rolling_std_12h"
]

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

station_features = [
    "StationId"
]

full_features = list(
    dict.fromkeys(
        current_features
        + lag_features
        + rolling_features
        + calendar_features
        + event_features
        + station_features
    )
)


# ------------------------------------------------------------
# 6. VERIFY FEATURES
# ------------------------------------------------------------

for feature_set_name, feature_list in [
    ("Current full model", full_features),
    ("Phase 14 candidate", candidate_features)
]:

    missing = [
        feature
        for feature in feature_list
        if feature not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing features for {feature_set_name}:\n"
            + "\n".join(missing)
        )


# ------------------------------------------------------------
# 7. LOAD TUNED RANDOM FOREST ARTIFACT
# ------------------------------------------------------------

saved_rf = joblib.load(
    TUNED_RF_PATH
)


# ------------------------------------------------------------
# 8. EXTRACT RANDOM FOREST HYPERPARAMETERS
# ------------------------------------------------------------

rf_default_params = (
    RandomForestRegressor(
        random_state=42
    ).get_params()
)

rf_param_names = set(
    rf_default_params.keys()
)


def extract_rf_params(obj):
    """
    Find Random Forest parameters from either:
    - RandomForestRegressor
    - Pipeline
    - nested estimator
    """

    # Direct RF
    if isinstance(
        obj,
        RandomForestRegressor
    ):
        return obj.get_params()

    # Pipeline / estimator with steps
    if hasattr(obj, "named_steps"):

        for _, step in obj.named_steps.items():

            params = extract_rf_params(step)

            if params is not None:
                return params

    # Generic steps
    if hasattr(obj, "steps"):

        for _, step in obj.steps:

            params = extract_rf_params(step)

            if params is not None:
                return params

    return None


actual_rf_params = extract_rf_params(
    saved_rf
)

if actual_rf_params is None:

    # Fallback: extract matching parameter names
    saved_params = (
        saved_rf.get_params()
        if hasattr(
            saved_rf,
            "get_params"
        )
        else {}
    )

    actual_rf_params = {}

    for key, value in saved_params.items():

        clean_key = key.split(
            "__"
        )[-1]

        if clean_key in rf_param_names:
            actual_rf_params[
                clean_key
            ] = value


# ------------------------------------------------------------
# 9. ENSURE REPRODUCIBILITY
# ------------------------------------------------------------

actual_rf_params[
    "random_state"
] = 42

actual_rf_params[
    "n_jobs"
] = -1


print("=" * 70)
print("PHASE 14.17 — FINAL FUTURE-TEST COMPARISON")
print("=" * 70)

print(
    "Candidate:",
    candidate_name
)

print(
    "Candidate features:",
    len(candidate_features)
)

print(
    "Current full-model features:",
    len(full_features)
)

print(
    "\nRF configuration:"
)

print(
    "n_estimators:",
    actual_rf_params.get(
        "n_estimators"
    )
)

print(
    "max_depth:",
    actual_rf_params.get(
        "max_depth"
    )
)

print(
    "min_samples_split:",
    actual_rf_params.get(
        "min_samples_split"
    )
)

print(
    "min_samples_leaf:",
    actual_rf_params.get(
        "min_samples_leaf"
    )
)

print(
    "max_features:",
    actual_rf_params.get(
        "max_features"
    )
)


# ------------------------------------------------------------
# 10. CREATE EXACT SAME 80/20 CHRONOLOGICAL SPLIT
#
# Phase 14.13 used:
#   first 80% = historical training
#   final 20% = future portion
#
# We reproduce that same logic here.
# ------------------------------------------------------------

split_index = int(
    len(df) * 0.80
)

train_df = df.iloc[
    :split_index
].copy()

test_df = df.iloc[
    split_index:
].copy()


# ------------------------------------------------------------
# 11. PREPARE DATA
# ------------------------------------------------------------

def prepare_xy(
    data,
    feature_list
):

    X = data[
        feature_list
    ].copy()

    y = pd.to_numeric(
        data[TARGET],
        errors="coerce"
    )

    return X, y


# ------------------------------------------------------------
# 12. BUILD LEAKAGE-SAFE MODEL PIPELINE
# ------------------------------------------------------------

def build_rf_pipeline(
    feature_list
):

    categorical_features = [
        feature
        for feature in feature_list
        if (
            df[feature].dtype == "object"
            or
            str(
                df[feature].dtype
            ).startswith("category")
        )
    ]

    numeric_features = [
        feature
        for feature in feature_list
        if feature not in categorical_features
    ]

    transformers = []

    # Numeric
    if numeric_features:

        numeric_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            )
        ])

        transformers.append(
            (
                "numeric",
                numeric_pipeline,
                numeric_features
            )
        )

    # Categorical
    if categorical_features:

        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ])

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    rf = RandomForestRegressor(
        **actual_rf_params
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            rf
        )
    ])


# ------------------------------------------------------------
# 13. EVALUATE ONE MODEL
# ------------------------------------------------------------

def evaluate_model(
    model_name,
    feature_list
):

    print("\n" + "=" * 60)
    print(
        f"TRAINING: {model_name}"
    )
    print(
        f"Features: {len(feature_list)}"
    )
    print("=" * 60)

    X_train, y_train = prepare_xy(
        train_df,
        feature_list
    )

    X_test, y_test = prepare_xy(
        test_df,
        feature_list
    )

    # Remove only rows with invalid targets.
    # Predictor missing values are handled by the pipeline.
    train_mask = np.isfinite(
        y_train.to_numpy()
    )

    test_mask = np.isfinite(
        y_test.to_numpy()
    )

    X_train = X_train.loc[
        train_mask
    ]

    y_train = y_train.loc[
        train_mask
    ]

    X_test = X_test.loc[
        test_mask
    ]

    y_test = y_test.loc[
        test_mask
    ]

    print(
        "Training rows:",
        len(X_train)
    )

    print(
        "Test rows:",
        len(X_test)
    )

    pipeline = build_rf_pipeline(
        feature_list
    )

    pipeline.fit(
        X_train,
        y_train
    )

    predictions = pipeline.predict(
        X_test
    )

    actual = y_test.to_numpy()
    predictions = np.asarray(
        predictions
    )

    valid = (
        np.isfinite(actual)
        &
        np.isfinite(predictions)
    )

    actual = actual[
        valid
    ]

    predictions = predictions[
        valid
    ]

    mae = mean_absolute_error(
        actual,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predictions
        )
    )

    r2 = r2_score(
        actual,
        predictions
    )

    accuracy_10 = (
        np.mean(
            np.abs(
                actual - predictions
            ) <= 10
        ) * 100
    )

    accuracy_20 = (
        np.mean(
            np.abs(
                actual - predictions
            ) <= 20
        ) * 100
    )

    return {
        "Model": model_name,
        "Feature Count": len(
            feature_list
        ),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2),
        "Accuracy ±10 AQI": float(
            accuracy_10
        ),
        "Accuracy ±20 AQI": float(
            accuracy_20
        ),
        "Valid Test Records": int(
            len(actual)
        )
    }


# ------------------------------------------------------------
# 14. EVALUATE BOTH MODELS
# ------------------------------------------------------------

current_result = evaluate_model(
    "Current Tuned Random Forest",
    full_features
)

candidate_result = evaluate_model(
    "Phase 14 — 7 Feature Candidate",
    candidate_features
)


# ------------------------------------------------------------
# 15. BUILD COMPARISON TABLE
# ------------------------------------------------------------

comparison = pd.DataFrame([
    current_result,
    candidate_result
])

comparison = comparison.sort_values(
    "RMSE",
    ascending=True
).reset_index(drop=True)

print("\n" + "=" * 70)
print("PHASE 14.17 — FUTURE TEST RESULTS")
print("=" * 70)

display(
    comparison
)


# ------------------------------------------------------------
# 16. CALCULATE DIFFERENCES
# ------------------------------------------------------------

champion = current_result
candidate = candidate_result

rmse_change = (
    candidate["RMSE"]
    - champion["RMSE"]
)

mae_change = (
    candidate["MAE"]
    - champion["MAE"]
)

accuracy_change = (
    candidate["Accuracy ±10 AQI"]
    - champion["Accuracy ±10 AQI"]
)

if rmse_change < 0:
    preliminary_decision = (
        "Candidate has lower RMSE on the "
        "same future test period."
    )
elif rmse_change > 0:
    preliminary_decision = (
        "Current tuned Random Forest has "
        "lower RMSE on the same future test period."
    )
else:
    preliminary_decision = (
        "Both configurations have identical RMSE."
    )


print("\n" + "=" * 70)
print("COMPARISON")
print("=" * 70)

print(
    "Candidate RMSE change:",
    round(rmse_change, 6)
)

print(
    "Candidate MAE change:",
    round(mae_change, 6)
)

print(
    "Candidate ±10 accuracy change:",
    round(accuracy_change, 4),
    "percentage points"
)

print(
    "\nPreliminary result:",
    preliminary_decision
)


# ------------------------------------------------------------
# 17. TEST-PERIOD DETAILS
# ------------------------------------------------------------

test_start = test_df[
    "Datetime"
].min()

test_end = test_df[
    "Datetime"
].max()

details = {
    "phase": "14",
    "step": "14.17",

    "split_method": (
        "Chronological 80/20 split"
    ),

    "split_index": int(
        split_index
    ),

    "total_rows": int(
        len(df)
    ),

    "training_rows": int(
        len(train_df)
    ),

    "test_rows": int(
        len(test_df)
    ),

    "test_start": str(
        test_start
    ),

    "test_end": str(
        test_end
    ),

    "stations": int(
        df["StationId"].nunique()
    ),

    "target": TARGET,

    "current_model": current_result,

    "candidate_model": candidate_result,

    "rmse_change": float(
        rmse_change
    ),

    "mae_change": float(
        mae_change
    ),

    "accuracy_plus_minus_10_change": float(
        accuracy_change
    ),

    "preliminary_decision":
        preliminary_decision,

    "status": (
        "TEST COMPLETED — "
        "FINAL PROMOTION DECISION RESERVED "
        "FOR PHASE 14.18"
    )
}


# ------------------------------------------------------------
# 18. SAVE RESULTS
# ------------------------------------------------------------

comparison.to_csv(
    SAVE_PATH,
    index=False
)

with open(
    DETAIL_PATH,
    "w"
) as f:
    json.dump(
        details,
        f,
        indent=4
    )

print("\n" + "=" * 70)
print("PHASE 14.17 COMPLETED")
print("=" * 70)

print(
    "Comparison saved:",
    SAVE_PATH
)

print(
    "Test details saved:",
    DETAIL_PATH
)

print(
    "Future test period:",
    test_start,
    "to",
    test_end
)

print("=" * 70)

In [ ]:
# ============================================================
# PHASE 14.18 — FORMAL FEATURE/MODEL DECISION
#
# Decision basis:
#   Same untouched future test period
#   Primary metric  : RMSE
#   Secondary       : MAE
#   Supporting      : Accuracy ±10 AQI
#
# The Phase 14 candidate is NOT promoted because it does not
# outperform the current tuned Random Forest.
# ============================================================

import os
import json
import pandas as pd

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_17_future_test_comparison.csv"
)

DETAIL_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_17_future_test_details.json"
)

DECISION_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase14_18_final_decision.json"
)

# ------------------------------------------------------------
# 2. VERIFY
# ------------------------------------------------------------

if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(
        f"Phase 14.17 results not found:\n{RESULTS_PATH}"
    )

# ------------------------------------------------------------
# 3. LOAD 14.17 RESULTS
# ------------------------------------------------------------

results = pd.read_csv(
    RESULTS_PATH
)

required_columns = [
    "Model",
    "Feature Count",
    "MAE",
    "RMSE",
    "R2",
    "Accuracy ±10 AQI",
    "Accuracy ±20 AQI",
    "Valid Test Records"
]

missing = [
    col
    for col in required_columns
    if col not in results.columns
]

if missing:
    raise ValueError(
        "Missing expected columns:\n"
        + "\n".join(missing)
    )

# ------------------------------------------------------------
# 4. IDENTIFY THE TWO CONFIGURATIONS
# ------------------------------------------------------------

current_name = (
    "Current Tuned Random Forest"
)

candidate_name = (
    "Phase 14 — 7 Feature Candidate"
)

current = results[
    results["Model"] == current_name
]

candidate = results[
    results["Model"] == candidate_name
]

if len(current) != 1:
    raise ValueError(
        "Could not uniquely identify current champion."
    )

if len(candidate) != 1:
    raise ValueError(
        "Could not uniquely identify Phase 14 candidate."
    )

current = current.iloc[0]
candidate = candidate.iloc[0]

# ------------------------------------------------------------
# 5. PRIMARY DECISION — RMSE
# ------------------------------------------------------------

current_rmse = float(
    current["RMSE"]
)

candidate_rmse = float(
    candidate["RMSE"]
)

current_mae = float(
    current["MAE"]
)

candidate_mae = float(
    candidate["MAE"]
)

current_acc10 = float(
    current["Accuracy ±10 AQI"]
)

candidate_acc10 = float(
    candidate["Accuracy ±10 AQI"]
)

rmse_difference = (
    candidate_rmse - current_rmse
)

mae_difference = (
    candidate_mae - current_mae
)

accuracy_difference = (
    candidate_acc10 - current_acc10
)

# Candidate is promoted ONLY if it has lower RMSE.
candidate_promoted = (
    candidate_rmse < current_rmse
)

if candidate_promoted:
    selected_model = (
        "Phase 14 — 7 Feature Candidate"
    )
    selected_feature_count = int(
        candidate["Feature Count"]
    )
    decision = (
        "Phase 14 candidate outperformed the "
        "current tuned Random Forest on the "
        "same untouched future test period."
    )
else:
    selected_model = current_name
    selected_feature_count = int(
        current["Feature Count"]
    )
    decision = (
        "Current tuned Random Forest retained. "
        "Phase 14 candidate did not outperform it "
        "on the same untouched future test period."
    )

# ------------------------------------------------------------
# 6. SAVE FORMAL DECISION
# ------------------------------------------------------------

decision_record = {
    "phase": "14",
    "step": "14.18",

    "evaluation_protocol": (
        "Same untouched chronological future test period"
    ),

    "primary_metric": "RMSE",

    "secondary_metrics": [
        "MAE",
        "R2",
        "Accuracy ±10 AQI",
        "Accuracy ±20 AQI"
    ],

    "current_model": {
        "name": current_name,
        "feature_count": int(
            current["Feature Count"]
        ),
        "MAE": current_mae,
        "RMSE": current_rmse,
        "R2": float(current["R2"]),
        "accuracy_plus_minus_10": current_acc10,
        "accuracy_plus_minus_20": float(
            current["Accuracy ±20 AQI"]
        )
    },

    "phase14_candidate": {
        "name": candidate_name,
        "feature_count": int(
            candidate["Feature Count"]
        ),
        "MAE": candidate_mae,
        "RMSE": candidate_rmse,
        "R2": float(candidate["R2"]),
        "accuracy_plus_minus_10": candidate_acc10,
        "accuracy_plus_minus_20": float(
            candidate["Accuracy ±20 AQI"]
        )
    },

    "candidate_vs_current": {
        "rmse_difference": rmse_difference,
        "mae_difference": mae_difference,
        "accuracy_plus_minus_10_difference":
            accuracy_difference
    },

    "candidate_promoted": bool(
        candidate_promoted
    ),

    "selected_model": selected_model,

    "selected_feature_count":
        selected_feature_count,

    "decision": decision,

    "status": (
        "SELECTED FOR CONTINUED VALIDATION — "
        "NOT YET FINAL DEPLOYMENT"
    )
}

with open(
    DECISION_PATH,
    "w"
) as f:
    json.dump(
        decision_record,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 7. DISPLAY
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 14.18 — FORMAL DECISION")
print("=" * 70)

print(
    "Selected model:",
    selected_model
)

print(
    "Selected feature count:",
    selected_feature_count
)

print(
    "\nCurrent RF RMSE:",
    round(current_rmse, 6)
)

print(
    "Candidate RMSE:",
    round(candidate_rmse, 6)
)

print(
    "Candidate RMSE difference:",
    round(rmse_difference, 6)
)

print(
    "\nCurrent RF ±10 accuracy:",
    round(current_acc10, 4),
    "%"
)

print(
    "Candidate ±10 accuracy:",
    round(candidate_acc10, 4),
    "%"
)

print(
    "Candidate promoted:",
    candidate_promoted
)

print(
    "\nDecision:"
)

print(decision)

print(
    "\nSaved:"
)

print(DECISION_PATH)

print("=" * 70)
print("PHASE 14.18 COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 14.19 — FINAL PHASE 14 CHECKPOINT
# Artifact inventory + final status
# ============================================================

import os
import json
import pandas as pd
from datetime import datetime

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

REPORT_DIR = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports"
)

CHECKPOINT_DIR = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "checkpoints"
)

MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "models"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. REQUIRED PHASE 14 ARTIFACTS
# ------------------------------------------------------------

artifacts = {
    "14.9_targeted_ablation":
        os.path.join(
            REPORT_DIR,
            "phase14_9_targeted_ablation_results.csv"
        ),

    "14.10_ablation_leaderboard":
        os.path.join(
            REPORT_DIR,
            "phase14_10_ablation_leaderboard.csv"
        ),

    "14.11_candidate_champion":
        os.path.join(
            REPORT_DIR,
            "phase14_11_candidate_champion_comparison.csv"
        ),

    "14.12_candidate_config":
        os.path.join(
            REPORT_DIR,
            "phase14_12_candidate_feature_config.json"
        ),

    "14.13_feature_importance":
        os.path.join(
            REPORT_DIR,
            "phase14_13_feature_importance.csv"
        ),

    "14.13_group_importance":
        os.path.join(
            REPORT_DIR,
            "phase14_13_feature_group_importance.csv"
        ),

    "14.14_pruning_candidates":
        os.path.join(
            REPORT_DIR,
            "phase14_14_pruning_candidates.json"
        ),

    "14.15_pruning_results":
        os.path.join(
            REPORT_DIR,
            "phase14_15_pruning_results.csv"
        ),

    "14.16_final_candidate":
        os.path.join(
            REPORT_DIR,
            "phase14_16_final_candidate_config.json"
        ),

    "14.17_future_test":
        os.path.join(
            REPORT_DIR,
            "phase14_17_future_test_comparison.csv"
        ),

    "14.17_test_details":
        os.path.join(
            REPORT_DIR,
            "phase14_17_future_test_details.json"
        ),

    "14.18_final_decision":
        os.path.join(
            REPORT_DIR,
            "phase14_18_final_decision.json"
        ),

    "tuned_random_forest":
        os.path.join(
            MODEL_DIR,
            "phase13B_tuned_random_forest.joblib"
        )
}

# ------------------------------------------------------------
# 3. CHECK ARTIFACTS
# ------------------------------------------------------------

artifact_status = []

for name, path in artifacts.items():

    exists = os.path.exists(path)

    artifact_status.append({
        "Artifact": name,
        "Exists": exists,
        "Path": path
    })

artifact_df = pd.DataFrame(
    artifact_status
)

# ------------------------------------------------------------
# 4. LOAD FINAL 14.17 RESULTS
# ------------------------------------------------------------

future_test_path = artifacts[
    "14.17_future_test"
]

future_test = pd.read_csv(
    future_test_path
)

current_row = future_test[
    future_test["Model"]
    == "Current Tuned Random Forest"
]

candidate_row = future_test[
    future_test["Model"]
    == "Phase 14 — 7 Feature Candidate"
]

if len(current_row) != 1:
    raise ValueError(
        "Current Tuned Random Forest result "
        "could not be uniquely identified."
    )

if len(candidate_row) != 1:
    raise ValueError(
        "Phase 14 candidate result "
        "could not be uniquely identified."
    )

current_row = current_row.iloc[0]
candidate_row = candidate_row.iloc[0]

# ------------------------------------------------------------
# 5. FINAL PHASE 14 SUMMARY
# ------------------------------------------------------------

final_summary = {
    "project":
        "Delhi AQI t+1 Hour Forecasting",

    "phase":
        "Phase 14 — Feature Optimization & Ablation",

    "completed_until":
        "14.19",

    "phase_status":
        "COMPLETED",

    "next_phase":
        "Phase 15",

    "target":
        "AQI_t_plus_1",

    "final_station_scope":
        10,

    "selected_model":
        "Current Tuned Random Forest",

    "selected_feature_count":
        int(current_row["Feature Count"]),

    "final_future_test_metrics": {
        "MAE":
            float(current_row["MAE"]),

        "RMSE":
            float(current_row["RMSE"]),

        "R2":
            float(current_row["R2"]),

        "Accuracy ±10 AQI":
            float(current_row["Accuracy ±10 AQI"]),

        "Accuracy ±20 AQI":
            float(current_row["Accuracy ±20 AQI"]),

        "Valid Test Records":
            int(current_row["Valid Test Records"])
    },

    "phase14_candidate": {
        "name":
            "Importance >= 0.00050",

        "feature_count":
            int(candidate_row["Feature Count"]),

        "future_test_RMSE":
            float(candidate_row["RMSE"]),

        "future_test_MAE":
            float(candidate_row["MAE"]),

        "future_test_accuracy_plus_minus_10":
            float(candidate_row["Accuracy ±10 AQI"]),

        "promoted":
            False
    },

    "feature_optimization_result": {
        "candidate":
            "7-feature Importance >= 0.00050",

        "decision":
            "Not promoted",

        "reason":
            "Higher RMSE, higher MAE, and lower ±10 AQI accuracy than the current tuned Random Forest on the same untouched future test period."
    },

    "validation_method":
        "Time-aware / leakage-free temporal validation",

    "artifact_inventory":
        artifact_status,

    "timestamp":
        datetime.now().isoformat()
}

# ------------------------------------------------------------
# 6. SAVE FINAL PHASE 14 RECORD
# ------------------------------------------------------------

FINAL_CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "phase14_final_checkpoint.json"
)

with open(
    FINAL_CHECKPOINT_PATH,
    "w"
) as f:
    json.dump(
        final_summary,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 7. UPDATE GENERAL SESSION STATUS
# ------------------------------------------------------------

SESSION_STATUS_PATH = os.path.join(
    CHECKPOINT_DIR,
    "session_status.json"
)

session_status = {
    "project":
        "Delhi AQI t+1 Hour Forecasting",

    "completed_phase":
        "Phase 14",

    "completed_until":
        "14.19",

    "resume_from":
        "Phase 15",

    "current_model":
        "Current Tuned Random Forest",

    "feature_count":
        int(current_row["Feature Count"]),

    "RMSE":
        float(current_row["RMSE"]),

    "MAE":
        float(current_row["MAE"]),

    "accuracy_plus_minus_10":
        float(current_row["Accuracy ±10 AQI"]),

    "status":
        "Phase 14 COMPLETED — READY FOR PHASE 15",

    "timestamp":
        datetime.now().isoformat()
}

with open(
    SESSION_STATUS_PATH,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 8. DISPLAY FINAL STATUS
# ------------------------------------------------------------

print("=" * 75)
print("PHASE 14.19 — FINAL PHASE 14 STATUS")
print("=" * 75)

print("Phase status       : COMPLETED")
print("Next phase         : Phase 15")
print("Target             : AQI_t_plus_1")
print("Stations           : 10")

print("\nSelected model:")
print("  Current Tuned Random Forest")

print("\nFinal feature count:")
print(
    int(current_row["Feature Count"])
)

print("\nFinal future-test metrics:")
print(
    f"  MAE               : "
    f"{current_row['MAE']:.6f}"
)

print(
    f"  RMSE              : "
    f"{current_row['RMSE']:.6f}"
)

print(
    f"  R2                : "
    f"{current_row['R2']:.6f}"
)

print(
    f"  Accuracy ±10 AQI  : "
    f"{current_row['Accuracy ±10 AQI']:.4f}%"
)

print(
    f"  Accuracy ±20 AQI  : "
    f"{current_row['Accuracy ±20 AQI']:.4f}%"
)

print("\nPhase 14 candidate:")
print(
    "  7-feature candidate promoted: FALSE"
)

print("\nArtifacts found:")
print(
    f"  {artifact_df['Exists'].sum()} / "
    f"{len(artifact_df)}"
)

display(
    artifact_df
)

print("\nFinal checkpoint:")
print(FINAL_CHECKPOINT_PATH)

print("\nSession status:")
print(SESSION_STATUS_PATH)

print("=" * 75)
print("PHASE 14.19 COMPLETED")
print("SAFE TO DISCONNECT COLAB RUNTIME")
print("NEXT: PHASE 15")
print("=" * 75)